In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from romlab import FOM, AdaptiveSampling, docker_solver

DATA = (ROOT / "data/2d/lambda-beta-adaptive").resolve()
FIELD = "c-trace"      # "velocity", "c-trace" or "mesh_coor2": the only field loaded, trained and stored
ACQ = "lhs"
SEED = 1
MAX_SAMPLES = 60

FILES = {"velocity": "snapshots", "c-trace": "c-trace", "mesh_coor2": "mesh_coor2"}
fom = FOM(
    data_folder=DATA,
    filenames_train=[f"{FILES[FIELD]}.txt", "parameters.txt"],
    filenames_test=[f"{FILES[FIELD]}_test.txt", "parameters_test.txt"],
    param_cols=[0, 1], fields=(FIELD,), seed=42)
fom.info()
initial = fom.parameters_train[:, :2].copy()

In [ ]:
solver = docker_solver(str(DATA), template_row=fom.parameters_train[0], param_cols=fom.param_cols, fields=(FIELD,))
adaptive = AdaptiveSampling(fom, solver, eps=1e-6, field=FIELD, acq=ACQ, n_candidates=50_000, plots=False, seed=SEED)
history = adaptive.run(max_samples=MAX_SAMPLES)

In [ ]:
tag = f"{FIELD}_{ACQ}_{SEED}"
results = DATA / "results"
results.mkdir(exist_ok=True)

rows = [[h["n_train"], h["nmodes"], h["error"], h["max_std"], *h.get("mu", [np.nan, np.nan]), h.get("std", np.nan)]
        for h in history]
cols = [("M", 5, "d"), ("r", 5, "d"), ("mean_error", 14, ".6e"), ("max_std", 14, ".6e"),
        ("next_lambda", 13, ".6f"), ("next_beta", 13, ".6f"), ("next_std", 14, ".6e")]
names = " ".join(f"{n:>{w}}" for n, w, _ in cols)  # header names right-aligned over their columns
np.savetxt(results / f"{tag}_results.txt", rows, fmt=[f"%{w}{s}" for _, w, s in cols], comments="",
           header=f"# field = {FIELD}, acq = {ACQ}, seed = {SEED}\n#{names[1:]}")

In [ ]:
added = fom.parameters_train[len(initial):, :2]
M = [h["n_train"] for h in history]

plt.rcParams.update({"font.size": 14, "axes.labelsize": 16, "xtick.labelsize": 13, "ytick.labelsize": 13})
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
ax.scatter(*fom.parameters_test[:, :2].T, s=4, c="lightgray", label="test")
ax.scatter(*initial.T, c="k", marker="s", label="initial LHS")
sc = ax.scatter(*added.T, c=np.arange(1, len(added) + 1), cmap="viridis", label=f"added ({ACQ})")
fig.colorbar(sc, ax=ax, label="iteration")
ax.set(xlabel=r"$\lambda$", ylabel=r"$\beta$")
ax.legend(loc="upper right", fontsize=10)

axes[1].semilogy(M, [h["error"] for h in history], "o-")
axes[1].set(xlabel="training samples $M$", ylabel="mean relative error", title=f"{FIELD}, acq = {ACQ}")
axes[2].semilogy(M, [h["max_std"] for h in history], "o-")
axes[2].set(xlabel="training samples $M$", ylabel="max predictive std")
plt.tight_layout()
plt.show()